# 01 · SDRS exploratory analysis: the recurring-defect long tail

**Corpus:** FAA Service Difficulty Reports pulled through `ingest/sdrs.py` (query-form export, 2024-01 → 2025-12), plus NASA ASRS maintenance narratives. Every row in Postgres carries the `ingest_runs` id of the payload it came from.

**Question this notebook answers:** how often does a chapter reappear on the same tail after it was already reported, and how concentrated is that behaviour? That is the pattern SIYANA's DALEEL module is built to catch.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sqlalchemy import text
from services.common.db import engine
pd.set_option('display.max_colwidth', 120)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
with engine.connect() as c:
    counts = pd.read_sql(text("select source, count(*) n, min(occurred_at) first, max(occurred_at) last from snags group by source order by n desc"), c)
    runs = pd.read_sql(text("select source, count(*) runs, sum(rows) rows_exported from ingest_runs group by source"), c)
display(counts); display(runs)

,source,n,first,last
0,sdrs,133296,2024-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00
1,asrs,18538,2012-01-01 00:00:00+00:00,2022-03-01 00:00:00+00:00


,source,runs,rows_exported
0,opensky,1,264
1,asrs,3,18538
2,sdrs,107,134690
3,cmapss,1,160359
4,mvtec,1,1504


In [2]:
with engine.connect() as c:
    sdrs = pd.read_sql(text('''
        select s.id, s.tail, s.occurred_at::date as day, s.ata_code, left(s.ata_code,2) as chapter, s.aircraft_type, s.defect_type,
               d.severity, s.raw_text
        from snags s join defect_events d on d.snag_id = s.id
        where s.source = 'sdrs' and s.ata_code is not null'''), c)
    chapters = pd.read_sql(text("select code, chapter, title from ata_chapters where code like '%00'"), c).set_index('chapter')['title']
sdrs['day'] = pd.to_datetime(sdrs['day'])
print(f"{len(sdrs):,} SDRS records with an ATA code, {sdrs['tail'].nunique():,} registrations, {sdrs.aircraft_type.nunique()} aircraft families")
sdrs.sample(5, random_state=1)[['tail','day','ata_code','aircraft_type','defect_type','severity','raw_text']]

133,296 SDRS records with an ATA code, 10,724 registrations, 360 aircraft families


,tail,day,ata_code,aircraft_type,defect_type,severity,raw_text
4549,N24202,2025-01-27,5340,B737,crack,S1,R/H VAPOR BARRIER I/B UPPER ATTACHED ANGLE CRACKED @ RBL 7 STA R/H VAPOR BARRIER I/B UPPER ATTACHED ANGLE CRACKED @ ...
69144,N17126,2025-04-08,3350,B757,crack,S1,(B757-53-3-0564) L/H FUSELAGE LIGHT LENS CRACKED AT AFT OVER WIN G HATCH DOT 140-065
11474,N337UP,2024-01-15,5210,B767,other,S1,**SUPPLEMENTAL SDR**COVER TO LOCK ON CABIN DOOR WILL NOT STOW TO DOWN POSITION REMOVED DEBRIS FROM HINGE REF IPC 52-...
87963,N714US,2024-10-03,5210,A319,other,S1,AIRCRAFT WAS NOT GROUNDED: AFT LEFT-HAND PASSENGER/CREW DOOR EMERGENCY CYLINDER ACCUMULATOR NEEDS SERVICED AS REQUIR...
33815,N122UP,2024-01-09,5300,A300F4,other,S1,FR 47 RH SPLICE FITTING HAS 2 HOLES DAMAGED AT THE BACK OF HOLE. AREAS MARKED IN RED. ROTO PERFORMED ON FRAME 47 RH ...


In [3]:
top = sdrs.chapter.value_counts().head(20).rename(index=lambda ch: f"{ch} {chapters.get(ch,'')}")
fig, ax = plt.subplots(figsize=(9,6))
top[::-1].plot.barh(ax=ax, color='#E8A317')
ax.set_title('Top 20 ATA chapters in SDRS, 2024–2025'); ax.set_xlabel('reports')
fig.tight_layout(); fig.savefig('../docs/figures/eda_top_chapters.png'); plt.close(fig)
top.to_frame('reports')

,reports
chapter,
53 Fuselage,51892
33 Lights,17097
52 Doors,11705
57 Wings,10924
25 Equipment / Furnishings,10911
55 Stabilizers,4406
21 Air Conditioning,4262
32 Landing Gear,2907
27 Flight Controls,2027


## Repeat chapters on the same tail

A **repeat** is a report on the same registration and the same 2-digit chapter within 90 days of an earlier one. This is the structural definition; DALEEL then decides whether two repeats are the *same defect signature* (semantic), which structure alone cannot tell you.

In [4]:
WINDOW = pd.Timedelta(days=90)
s = sdrs.dropna(subset=['tail']).sort_values(['tail','chapter','day'])
s['prev_day'] = s.groupby(['tail','chapter'])['day'].shift(1)
s['gap'] = s['day'] - s['prev_day']
s['repeat'] = s['gap'].le(WINDOW)
per_tail = s.groupby('tail').agg(reports=('id','size'), repeats=('repeat','sum'), chapters=('chapter','nunique'))
per_tail['any_repeat'] = per_tail.repeats > 0
share = per_tail.loc[per_tail.reports >= 3, 'any_repeat'].mean()
print(f"Tails with >= 3 reports: {int((per_tail.reports>=3).sum()):,}; share with at least one 90-day repeat: {share:.1%}")
print(f"Repeat reports overall: {int(s['repeat'].sum()):,} of {len(s):,} ({s['repeat'].mean():.1%})")
gap_days = s.loc[s['repeat'], 'gap'].dt.days
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(gap_days, bins=45, color='#4E8C6A'); ax.set_xlabel('days since previous report, same tail and chapter'); ax.set_ylabel('repeat reports')
ax.set_title('How soon a chapter comes back on the same tail (<= 90 days)')
fig.tight_layout(); fig.savefig('../docs/figures/eda_repeat_gap.png'); plt.close(fig)

Tails with >= 3 reports: 7,254; share with at least one 90-day repeat: 87.1%
Repeat reports overall: 81,237 of 132,595 (61.3%)


In [5]:
# The long tail: a few tail/chapter pairs carry most of the repeats.
pairs = s[s['repeat']].groupby(['tail','chapter']).size().sort_values(ascending=False)
cum = pairs.cumsum() / pairs.sum()
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(np.arange(1, len(cum)+1), cum.values, color='#C4342B')
ax.set_xscale('log'); ax.set_xlabel('tail × chapter pairs, ranked'); ax.set_ylabel('cumulative share of repeat reports')
ax.set_title('Repeat reports concentrate in a long tail of tail × chapter pairs')
k10 = int(np.searchsorted(cum.values, 0.5)) + 1
ax.axhline(0.5, ls='--', lw=0.8, color='#8FA8B5'); ax.axvline(k10, ls='--', lw=0.8, color='#8FA8B5')
fig.tight_layout(); fig.savefig('../docs/figures/eda_long_tail.png'); plt.close(fig)
print(f"{k10:,} of {len(pairs):,} tail×chapter pairs ({k10/len(pairs):.1%}) account for half of all repeat reports")
pairs.head(12).rename('repeats').to_frame().assign(title=lambda d: [chapters.get(ch,'') for _, ch in d.index])

1,594 of 15,858 tail×chapter pairs (10.1%) account for half of all repeat reports


,,repeats,title
tail,chapter,,
N392DA,53,115,Fuselage
N3739P,53,103,Fuselage
N397DA,53,101,Fuselage
N406LC,53,96,Fuselage
N860NW,53,92,Fuselage
N8322X,53,91,Fuselage
N284WN,53,91,Fuselage
N583UP,53,89,Fuselage
N391DA,53,87,Fuselage


In [6]:
# Repeat rate by chapter: where does rectification least often stick?
by_ch = s.groupby('chapter').agg(reports=('id','size'), repeat_rate=('repeat','mean')).query('reports >= 300').sort_values('repeat_rate', ascending=False)
by_ch['title'] = [chapters.get(ch,'') for ch in by_ch.index]
by_ch.head(15)

,reports,repeat_rate,title
chapter,,,
53,51814,0.881499,Fuselage
57,10921,0.720813,Wings
55,4395,0.619568,Stabilizers
52,11675,0.503812,Doors
33,17092,0.473496,Lights
78,715,0.448951,Exhaust
54,1545,0.438188,Nacelles / Pylons
25,10884,0.414462,Equipment / Furnishings
35,630,0.400000,Oxygen


## Why free text matters

The same repeat written differently. The rows below are pairs of repeats on one tail and chapter; token overlap is low even when the defect is plainly the same. Keyword search misses these; embeddings plus a judge do not.

In [7]:
import re
def toks(t): return set(re.findall(r'[A-Z]{3,}', t.upper()))
ex = []
for (tail, ch), g in s[s['repeat']].groupby(['tail','chapter']):
    g = g.sort_values('day')
    for i in range(1, len(g)):
        a, b = g.iloc[i-1], g.iloc[i]
        ov = len(toks(a.raw_text) & toks(b.raw_text)) / max(1, len(toks(a.raw_text) | toks(b.raw_text)))
        ex.append((tail, ch, a.day.date(), b.day.date(), round(ov,2), a.raw_text[:110], b.raw_text[:110]))
ex = pd.DataFrame(ex, columns=['tail','chapter','first','repeat','jaccard','first_text','repeat_text'])
print(f"median Jaccard token overlap between a report and its 90-day repeat: {ex.jaccard.median():.2f}")
ex.sort_values('jaccard').head(8)

median Jaccard token overlap between a report and its 90-day repeat: 0.35


,tail,chapter,first,repeat,jaccard,first_text,repeat_text
49183,N829UA,53,2024-11-27,2024-12-02,0.0,AFT CABIN RT SIDE Y-765 CORRODED AT FR68 PER HAECO LCQ N/R 9296821/20821,ON TOP OF FLOOR SUPPORT +Y765 BETWEEN FR67 AND FR68 HAS CORROSION AROUND 2 FASTENER HOLES. RMVD AND RPLD FLOOR
56681,N88326,55,2025-03-04,2025-03-07,0.0,"PANEL 345 AR, RH HORIZONTAL TIP CAP, INNER NOSE RIB IS CRACKED.",326AL UPPER T/E APPEARS TO HAVE LIGHTNING EXIT
42061,N75853,25,2024-11-11,2024-11-17,0.0,"BOTH COFFEE MAKERS IN THE AFT GALLEY WERE SMOKING DURING T/O AN D CLB. FAS PULLED CBS FOUND SOURCE OF SMOKE ,","SLIDE ARM LIGHT INOP AT 1R PERFORMED R1 DOOR ARMING SYS PER AMM 52-11-00, CLEANED GIRTBAR TRACKED. OPS CKS GOO"
532,N11176,33,2024-11-30,2025-02-08,0.0,MULTIPLE EMERGENCY LIGHTS INOPERATIVE. REMOVED AND REPLACED BATTERY POWER SUPPLY. OPS CHECK SAT.,PHOTO-LUMINESCENT LIGHTING STRIP MISSING AT FRONT OF AIRCRAFT. INSTALLED MISSING LIGHT STRIP
45028,N7817J,33,2024-04-27,2024-07-25,0.0,FOUND RT AFT EMER LTS CB OUT. RESET W/CHRIS IN MX CONTROL 146124. EMER LTS OPERATE NORMALLY. INFO ONLY<[01] NO,EMERGENCY LIGHTS AT OVERWING EXITS (LH & RH SIDE) INOP<[01] REMOVED; REPLACED AND PERFORMED GOOD FUNCTIONAL TE
37684,N686AE,53,2024-06-28,2024-07-02,0.0,"53-21388- CENTER FUSE III, LH OMEGA BEAM CRACKED FRAME 36 AND 37.","AFT FUSELAGE BETWEEN THE FRAMES 57-58 ON RH, THE SKIN IS CORRODED FROM THE EXTERNAL AROUND THE POTABLE REFILL"
56683,N88326,57,2025-03-05,2025-03-06,0.0,RH WING FIXED LEADING EDGE OVER PYLON HAS CHAFE MARKS OUT OF LIMITS,RH AILERON QUADRANT ASSEMBLY IN THE RH MLG WHEEL WELL LOWER BRACKET IS CRACKED
3899,N150UP,52,2024-03-02,2025-02-07,0.0,"*SUPPLEMENTAL SDR* ASPSU FAILED OPS TEST R&R ASPSU 70WN PER A300 AMM 52-73-18, OPS CK GOOD.",L1 CREW DOOR EXTERNAL LOWER SKIN HAS GOUGES .89-25 PERFORMED HFEC ON BLENDED AREA. NO DEFECTS FOUND. IAW NTM 5


## Take-aways for the build

1. Repeats are common and concentrated: a small set of tail × chapter pairs carries half of them. The control room's heatmap is built to show exactly that concentration.
2. Structural chapters (53 fuselage, 57 wings, 52 doors) dominate the corpus; engine chapters are fewer but carry the highest operational cost, which is why the RUL module targets them.
3. Token overlap between a report and its repeat is low. That is the empirical case for semantic retrieval plus an LLM judge over keyword matching, and it is what `notebooks/03_recurrence_eval.ipynb` measures.